# Life Segment Clustering

Unsupervised discovery of player role archetypes from `life_segment_features`.

**Data:** 3 matches across 3 maps (annealing_iv · tumbleweed · outback_outback_edition), 2 318 life segments total.

**Goal:** cluster life segments into behavioural archetypes (defender, mid-fighter, rusher, etc.) using position-derived features, then write labels back to the database.

---

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import hdbscan

DB_PATH = '../match_analysis/metadata.db'

sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('Ready.')

## 2. Load & Inspect Raw Features

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)

df = conn.execute("""
    SELECT
        f.segment_id,
        m.map_slug,
        pts.team,
        -- region-based features (original)
        f.n_islands_visited,
        f.n_build_regions_visited,
        f.n_transitions,
        f.frac_time_home_island,
        f.frac_time_enemy_island,
        f.frac_time_neutral_island,
        f.frac_time_build,
        f.max_attack_depth,
        f.duration_s,
        f.time_to_first_departure_s,
        f.kills,
        f.deaths,
        f.kill_in_build,
        f.kill_on_enemy_island,
        f.wool_captures,
        f.ended_on_enemy_island,
        f.ended_in_build,
        -- node-path features (skeleton-level)
        f.visited_junction,
        f.frac_island_visits_with_junction,
        f.max_node_degree_visited,
        f.traversal_rate,
        f.avg_nodes_per_island_visit,
        f.died_at_endpoint,
        f.n_unique_corridors,
        f.position_entropy,
        f.dominant_node_frac
    FROM life_segment_features f
    JOIN life_segments ls ON f.segment_id = ls.segment_id
    JOIN matches mat ON mat.match_id = ls.match_id
    JOIN maps m ON m.map_id = mat.map_id
    LEFT JOIN player_team_segments pts
        ON pts.player_id = ls.player_id
        AND pts.match_id = ls.match_id
        AND pts.start_timestamp <= ls.start_timestamp
        AND (pts.end_timestamp IS NULL OR pts.end_timestamp >= ls.end_timestamp)
    ORDER BY f.segment_id
""").df()

conn.close()

print(f"Loaded {len(df):,} life segments")
print(f"Maps: {df['map_slug'].value_counts().to_dict()}")
df.head()

In [ ]:
# Null audit
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

## 3. Feature Engineering

In [ ]:
fe = df.copy()

# --- Imputation -------------------------------------------------------
# time_to_first_departure_s is NULL when the player never left home.
# Impute with duration_s so these lives read as "departed at the very end".
fe['time_to_first_departure_s'] = fe['time_to_first_departure_s'].fillna(fe['duration_s'])

# Node-path nulls: segments with zero island visits have no node data.
# Fill continuous node-path features with 0 and booleans with False.
fe['frac_island_visits_with_junction'] = fe['frac_island_visits_with_junction'].fillna(0.0)
fe['traversal_rate']                   = fe['traversal_rate'].fillna(0.0)
fe['avg_nodes_per_island_visit']       = fe['avg_nodes_per_island_visit'].fillna(0.0)
fe['position_entropy']                 = fe['position_entropy'].fillna(0.0)
fe['dominant_node_frac']               = fe['dominant_node_frac'].fillna(1.0)
fe['n_unique_corridors']               = fe['n_unique_corridors'].fillna(0)
fe['visited_junction']                 = fe['visited_junction'].fillna(False)
fe['max_node_degree_visited']          = fe['max_node_degree_visited'].fillna(1)
# died_at_endpoint is intentionally left as None for lives not ending on an island.

# --- Derived features (region-level) ----------------------------------
# Kill rate: kills per game-tick of life (safe divide)
fe['kill_rate'] = fe['kills'] / fe['duration_s'].clip(lower=1)

# Departure fraction: how quickly did the player leave their home island?
# 0 = left immediately, 1 = never left
fe['departure_frac'] = fe['time_to_first_departure_s'] / fe['duration_s'].clip(lower=1)
fe['departure_frac'] = fe['departure_frac'].clip(0, 1)

# Mobility rate: transitions per tick (how actively they moved)
fe['mobility_rate'] = fe['n_transitions'] / fe['duration_s'].clip(lower=1)

# Combat aggression: fraction of kills on enemy island vs total kills
fe['aggression'] = np.where(
    fe['kills'] > 0,
    fe['kill_on_enemy_island'] / fe['kills'],
    0.0
)

print("Region-level engineered features:")
fe[['kill_rate', 'departure_frac', 'mobility_rate', 'aggression']].describe()

print("\nNode-path features (null audit after imputation):")
node_cols = [
    'visited_junction', 'frac_island_visits_with_junction', 'max_node_degree_visited',
    'traversal_rate', 'avg_nodes_per_island_visit', 'died_at_endpoint',
    'n_unique_corridors', 'position_entropy', 'dominant_node_frac',
]
print(fe[node_cols].describe().T[['mean', 'std', 'min', 'max']].round(3))

## 4. Feature Matrix for Clustering

### Region-level features (original)

| Feature | Captures |
|---|---|
| `max_attack_depth` | How far toward objective the player pushed |
| `frac_time_home_island` | Defensive presence |
| `frac_time_enemy_island` | Offensive presence |
| `frac_time_build` | Bridge / mid-void fighting |
| `departure_frac` | How late (or never) they left home |
| `kill_rate` | Killing efficiency independent of life length |
| `aggression` | Whether kills happen on offense vs defense |
| `mobility_rate` | How actively they traversed the map |

### Node-path features (skeleton-level, new)

| Feature | Captures |
|---|---|
| `frac_island_visits_with_junction` | Fraction of island visits reaching interior junction nodes (degree ≥ 3); proxy for depth quality |
| `traversal_rate` | Fraction of island visits where the player moved enough to shift closest skeleton node; distinguishes active movers from static campers |
| `position_entropy` | Shannon entropy of node-visit distribution; high = roamer spread across many nodes, low = camper |
| `dominant_node_frac` | Fraction of all node appearances at the single most-visited node; inverse of entropy, direct camper signal |

**Why these four for clustering:**
`frac_island_visits_with_junction` refines `max_attack_depth` by capturing *consistency* of deep play, not just a single peak.
`traversal_rate` is orthogonal to time-fraction features: a player can spend lots of time on enemy island while barely moving.
`position_entropy` / `dominant_node_frac` are anti-correlated (one is redundant as a cluster feature) — include `position_entropy`.

In [ ]:
CLUSTER_FEATURES = [
    # --- Region-level (original 8) ---
    'max_attack_depth',
    'frac_time_home_island',
    'frac_time_enemy_island',
    'frac_time_build',
    'departure_frac',
    'kill_rate',
    'aggression',
    'mobility_rate',
    # --- Node-path features (new 3) ---
    'frac_island_visits_with_junction',   # depth quality: consistency of reaching interior
    'traversal_rate',                     # active movement vs static positioning
    'position_entropy',                   # spatial diversity: roamer vs camper
]

X_raw = fe[CLUSTER_FEATURES].values

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Feature matrix: {X.shape} ({len(CLUSTER_FEATURES)} features)")

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = fe[CLUSTER_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (11 features)')
plt.tight_layout()
plt.savefig('../output/clustering_corr.png', dpi=120)
plt.show()

## 5. PCA — Dimensionality Overview

In [ ]:
pca = PCA(n_components=len(CLUSTER_FEATURES))
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
evr = pca.explained_variance_ratio_
axes[0].bar(range(1, len(evr)+1), evr * 100, color='steelblue')
axes[0].plot(range(1, len(evr)+1), np.cumsum(evr) * 100, 'o-', color='tomato')
axes[0].axhline(80, color='grey', ls='--', lw=1)
axes[0].set_xlabel('PC')
axes[0].set_ylabel('Explained variance (%)')
axes[0].set_title('Scree plot')

# PC1 vs PC2 coloured by attack depth
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                     c=fe['max_attack_depth'], cmap='plasma',
                     s=8, alpha=0.6)
plt.colorbar(sc, ax=axes[1], label='attack depth')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('PC1 vs PC2 (colour = attack depth)')

plt.tight_layout()
plt.savefig('../output/clustering_pca.png', dpi=120)
plt.show()

print("Cumulative variance by PC:")
for i, v in enumerate(np.cumsum(evr) * 100, 1):
    print(f"  PC{i}: {v:.1f}%")

In [ ]:
# PCA loadings — which features drive each PC?
loadings = pd.DataFrame(
    pca.components_[:4].T,
    index=CLUSTER_FEATURES,
    columns=[f'PC{i+1}' for i in range(4)]
).round(3)
print("PCA loadings (top 4 PCs):")
loadings

## 6. KMeans — Elbow / Silhouette Scan

In [ ]:
K_RANGE = range(2, 10)
inertias, sil_scores = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels, sample_size=min(2000, len(X))))
    print(f"  k={k}  inertia={km.inertia_:.1f}  silhouette={sil_scores[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(K_RANGE), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow')
axes[1].plot(list(K_RANGE), sil_scores, 'o-', color='tomato')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette score'); axes[1].set_title('Silhouette')
plt.tight_layout()
plt.savefig('../output/clustering_kmeans_scan.png', dpi=120)
plt.show()

## 7. HDBSCAN Clustering

In [ ]:
# Use first 4 PCs (typically ~80% variance) for HDBSCAN — reduces noise sensitivity
N_PCS = 4
X_reduced = PCA(n_components=N_PCS, random_state=42).fit_transform(X)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=40,
    min_samples=10,
    cluster_selection_method='eom',
    metric='euclidean',
)
hdb_labels = clusterer.fit_predict(X_reduced)

n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise = (hdb_labels == -1).sum()
print(f"HDBSCAN: {n_clusters} clusters, {n_noise} noise points ({n_noise/len(hdb_labels)*100:.1f}%)")
pd.Series(hdb_labels).value_counts().sort_index()

## 8. Choose Final Clustering

Inspect both approaches and pick the one that produces the cleanest archetypes. Edit `FINAL_LABELS` below.

In [ ]:
# --- Pick the best k from the scan above and re-fit KMeans -------------
BEST_K = 5   # <-- adjust after reviewing elbow/silhouette plots

km_final = KMeans(n_clusters=BEST_K, n_init=30, random_state=42)
km_labels = km_final.fit_predict(X)

# --- Choose which clustering to use for labelling ----------------------
# Options: 'kmeans'  or  'hdbscan'
USE = 'kmeans'

if USE == 'hdbscan':
    FINAL_LABELS = hdb_labels
else:
    FINAL_LABELS = km_labels

fe = fe.copy()
fe['cluster_id'] = FINAL_LABELS

print(f"Using: {USE}")
print(fe['cluster_id'].value_counts().sort_index())

## 9. Cluster Characterisation

In [ ]:
PROFILE_COLS = [
    # Region-level
    'max_attack_depth', 'departure_frac',
    'frac_time_home_island', 'frac_time_enemy_island', 'frac_time_build',
    'kill_rate', 'aggression', 'mobility_rate',
    'kills', 'wool_captures', 'duration_s',
    # Node-path
    'frac_island_visits_with_junction', 'traversal_rate',
    'position_entropy', 'dominant_node_frac',
    'avg_nodes_per_island_visit', 'n_unique_corridors',
]

profile = fe.groupby('cluster_id')[PROFILE_COLS].mean().round(3)
profile['n'] = fe.groupby('cluster_id')['segment_id'].count()
profile['caps'] = fe.groupby('cluster_id')['wool_captures'].sum()
profile

In [ ]:
# Radar / spider chart for each cluster (now includes node-path features)
RADAR_FEATURES = [
    'max_attack_depth', 'frac_time_enemy_island', 'frac_time_build',
    'kill_rate', 'aggression', 'mobility_rate',
    'frac_island_visits_with_junction', 'traversal_rate', 'position_entropy',
]

# Normalise to [0,1] across clusters for radar readability
radar_data = profile[RADAR_FEATURES].copy()
radar_data = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(RADAR_FEATURES), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

cluster_ids = sorted(fe['cluster_id'].unique())
palette = sns.color_palette('tab10', n_colors=len(cluster_ids))

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})
for cid, color in zip(cluster_ids, palette):
    if cid == -1:
        continue
    vals = radar_data.loc[cid].tolist() + [radar_data.loc[cid].tolist()[0]]
    label = f"C{cid} (n={int(profile.loc[cid, 'n'])})"
    ax.plot(angles, vals, 'o-', lw=2, color=color, label=label)
    ax.fill(angles, vals, alpha=0.10, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n') for f in RADAR_FEATURES], size=8)
ax.set_yticklabels([])
ax.set_title('Cluster profiles (normalised, 9-axis)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15))
plt.tight_layout()
plt.savefig('../output/clustering_radar.png', dpi=120)
plt.show()

In [ ]:
# PCA scatter coloured by cluster
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (xi, yi, title) in zip(axes, [
    (0, 1, 'PC1 vs PC2'),
    (2, 3, 'PC3 vs PC4'),
]):
    for cid, color in zip(cluster_ids, palette):
        mask = fe['cluster_id'] == cid
        label = f'C{cid}' if cid != -1 else 'noise'
        alpha = 0.25 if cid == -1 else 0.55
        ax.scatter(X_pca[mask, xi], X_pca[mask, yi],
                   c=[color], s=10, alpha=alpha, label=label)
    ax.set_xlabel(f'PC{xi+1}')
    ax.set_ylabel(f'PC{yi+1}')
    ax.set_title(title)
    ax.legend(markerscale=2, fontsize=8)

plt.suptitle('Clusters in PCA space', fontsize=13)
plt.tight_layout()
plt.savefig('../output/clustering_pca_clusters.png', dpi=120)
plt.show()

In [ ]:
# Box plots: key features by cluster (region-level + node-path)
BOX_FEATURES = [
    'max_attack_depth', 'frac_time_home_island', 'frac_time_enemy_island',
    'kill_rate', 'departure_frac',
    'frac_island_visits_with_junction', 'traversal_rate', 'position_entropy',
]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, feat in zip(axes, BOX_FEATURES):
    data = [fe.loc[fe['cluster_id'] == cid, feat].values for cid in cluster_ids]
    bp = ax.boxplot(data, patch_artist=True, labels=[f'C{c}' for c in cluster_ids])
    for patch, color in zip(bp['boxes'], palette):
        patch.set_facecolor((*color[:3], 0.5))
    ax.set_title(feat.replace('_', '\n'), fontsize=9)
    ax.tick_params(axis='x', labelsize=8)
plt.suptitle('Feature distributions by cluster', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../output/clustering_boxplots.png', dpi=120)
plt.show()

## 10. Assign Archetype Labels

Edit the mapping below based on what you see in the cluster profiles.

In [ ]:
# Print a summary to help assign names
for cid in cluster_ids:
    if cid == -1:
        print(f"  C{cid:2d}  NOISE  n={int((fe['cluster_id']==-1).sum())}")
        continue
    row = profile.loc[cid]
    print(
        f"  C{cid}  n={int(row['n']):4d}  "
        f"depth={row['max_attack_depth']:.2f}  "
        f"home={row['frac_time_home_island']:.2f}  "
        f"enemy={row['frac_time_enemy_island']:.2f}  "
        f"build={row['frac_time_build']:.2f}  "
        f"kill_rate={row['kill_rate']:.3f}  "
        f"depart={row['departure_frac']:.2f}  "
        f"jxn_visits={row['frac_island_visits_with_junction']:.2f}  "
        f"traversal={row['traversal_rate']:.2f}  "
        f"entropy={row['position_entropy']:.2f}  "
        f"caps={int(row['caps'])}"
    )

In [ ]:
# --- Cluster profiles from execution: ---
#
#  C0  n= 253  depth=0.76  home=0.30  enemy=0.60  build=0.04  kill_rate=0.017  depart=0.28  caps=5
#    → deep-attacker: rushes deep, fights on enemy island, high kill rate
#
#  C1  n= 790  depth=0.69  home=0.40  enemy=0.33  build=0.05  kill_rate=0.005  depart=0.33  caps=8
#    → attacker: pushes deep with moderate home time, lower kill rate (objective focus)
#
#  C2  n=1021  depth=0.26  home=0.93  enemy=0.01  build=0.02  kill_rate=0.003  depart=0.84  caps=1
#    → defender: stays on home island, rarely pushes
#
#  C3  n= 242  depth=0.53  home=0.32  enemy=0.05  build=0.55  kill_rate=0.006  depart=0.32  caps=0
#    → bridge-fighter: most life spent in build/void regions between islands
#
#  C4  n=  12  depth=0.02  home=0.00  enemy=0.00  build=0.00  kill_rate=0.000  depart=0.83  caps=0
#    → outlier: near-zero presence in all zone types (neutral islands / edge cases)

CLUSTER_LABELS = {
    -1: 'unclassified',
    0:  'deep-attacker',   # high depth + enemy time + kill rate; fights on offense
    1:  'attacker',        # pushes deep, objective-focused, moderate home time
    2:  'defender',        # almost exclusively on home island, slow to depart
    3:  'bridge-fighter',  # most time in build/void regions, mid-map combat
    4:  'outlier',         # tiny cluster; neutral-island or edge-case lives
}

fe['cluster_label'] = fe['cluster_id'].map(CLUSTER_LABELS).fillna('unclassified')

print(fe.groupby(['cluster_id', 'cluster_label']).size().reset_index(name='n').to_string(index=False))

## 11. Write Labels Back to Database

In [ ]:
write_conn = duckdb.connect(DB_PATH)

# Register the result dataframe and batch-update via a join
labels_df = fe[['segment_id', 'cluster_id', 'cluster_label']].copy()
write_conn.register('_cluster_labels', labels_df)

write_conn.execute("""
    UPDATE life_segment_features AS f
    SET cluster_id    = l.cluster_id,
        cluster_label = l.cluster_label
    FROM _cluster_labels l
    WHERE f.segment_id = l.segment_id
""")

n_written = write_conn.execute(
    "SELECT COUNT(*) FROM life_segment_features WHERE cluster_label IS NOT NULL"
).fetchone()[0]

write_conn.close()
print(f"Wrote cluster labels for {n_written:,} segments.")

## 12. Cross-Map Archetype Distribution

Do the same archetypes appear consistently across all three maps?

In [ ]:
map_cluster = (
    fe.groupby(['map_slug', 'cluster_label'])
    .size()
    .rename('n')
    .reset_index()
)

# Compute percentage within each map
map_totals = map_cluster.groupby('map_slug')['n'].transform('sum')
map_cluster['pct'] = (map_cluster['n'] / map_totals * 100).round(1)

pivot = map_cluster.pivot(index='cluster_label', columns='map_slug', values='pct').fillna(0)
print("Archetype distribution by map (% of lives):")
print(pivot.round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
pivot.plot(kind='bar', ax=ax, rot=30)
ax.set_ylabel('% of lives')
ax.set_title('Archetype distribution by map')
ax.legend(title='map', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('../output/clustering_map_distribution.png', dpi=120)
plt.show()

In [ ]:
# Per-player dominant archetype
player_arch = (
    fe.groupby(['map_slug', 'team', 'cluster_label'])
    .agg(
        n_lives=('segment_id', 'count'),
        avg_depth=('max_attack_depth', 'mean'),
        total_kills=('kills', 'sum'),
        total_caps=('wool_captures', 'sum'),
    )
    .reset_index()
    .sort_values(['map_slug', 'cluster_label'])
)

print("Lives per archetype broken down by team and map:")
player_arch

## 13. Node-Path Feature Deep Dive

Examine how the skeleton-level metrics distribute across the archetypes identified above.
These are the operationalized findings from the corridor / junction analysis.

In [ ]:
NODE_FEATURES = [
    'frac_island_visits_with_junction',
    'traversal_rate',
    'position_entropy',
    'dominant_node_frac',
    'avg_nodes_per_island_visit',
    'n_unique_corridors',
]

node_profile = fe.groupby('cluster_label')[NODE_FEATURES].mean().round(3)
node_profile['n'] = fe.groupby('cluster_label')['segment_id'].count()

# Junction visitor rate per archetype
node_profile['pct_visited_junction'] = (
    fe.groupby('cluster_label')['visited_junction'].mean() * 100
).round(1)

# Endpoint-death rate (only lives that ended on an island)
island_deaths = fe[fe['died_at_endpoint'].notna()].copy()
endpoint_death_rate = (
    island_deaths.groupby('cluster_label')['died_at_endpoint'].mean() * 100
).round(1)
node_profile['pct_died_at_endpoint'] = endpoint_death_rate

print("Node-path profile by cluster archetype:")
node_profile

In [ ]:
# Visualise: junction visit rate and traversal rate by archetype
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels = node_profile.index.tolist()
x = np.arange(len(labels))

ax = axes[0]
ax.bar(x, node_profile['frac_island_visits_with_junction'], color='steelblue', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha='right')
ax.set_ylabel('Fraction of island visits')
ax.set_title('Junction penetration rate by archetype')

ax = axes[1]
ax.bar(x, node_profile['traversal_rate'], color='tomato', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha='right')
ax.set_ylabel('Fraction of island visits')
ax.set_title('Traversal rate by archetype')

ax = axes[2]
ax.bar(x, node_profile['position_entropy'], color='mediumseagreen', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha='right')
ax.set_ylabel('Shannon entropy (bits)')
ax.set_title('Position entropy by archetype')

plt.suptitle('Skeleton-level metrics by cluster archetype', fontsize=12)
plt.tight_layout()
plt.savefig('../output/clustering_node_metrics.png', dpi=120)
plt.show()

# Scatter: junction penetration vs traversal rate, coloured by archetype
fig, ax = plt.subplots(figsize=(8, 6))
arch_palette = dict(zip(sorted(fe['cluster_label'].unique()), sns.color_palette('tab10', 6)))
for label, grp in fe.groupby('cluster_label'):
    ax.scatter(
        grp['traversal_rate'], grp['frac_island_visits_with_junction'],
        c=[arch_palette[label]], s=8, alpha=0.35, label=label,
    )
ax.set_xlabel('Traversal rate (entry_node ≠ exit_node fraction)')
ax.set_ylabel('Frac island visits with junction')
ax.set_title('Traversal vs Junction Penetration')
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.savefig('../output/clustering_traversal_vs_junction.png', dpi=120)
plt.show()